# Pre-processing + feature reduction

## Set up

Import data/packages/libraries

In [ ]:
import sys

sys.path.append("../")
from src.config import BASE_PATH, SEED
from src.preprocess import transform_export_data_dev
import pandas as pd

In [ ]:
outcome_list = [
    "SERIOUS",
    "ANY",
    "PNEUMO",
    "CARDIAC_COMP",
    "DVT",
    "SEPSIS",
    "SSI",
    "UTI",
    "RENAL",
    "UNPLNREOP",
    "MORT",
]
cohort_list = [
    "cancer",
    "all",
]
x_cols = [
    # Pre-Op
    "AGE",
    "HEIGHT",
    "WEIGHT",
    "SEX",
    "HISPANIC",
    "RACE",
    "DIABETES",
    "HXCOPD",
    "HXCHF",
    "ASCITES",
    "BLEEDDIS",
    "TRANSFUS",
    "DIALYSIS",
    "HYPERMED",
    "VENTILAT",
    "SMOKE",
    "DISCANCR",
    "STEROID",
    "ASACLAS",
    "PRALBUM",
    "PRWBC",
    "PRHCT",
    "PRPLATE",
    # Surgical Characteristics
    "SURGINDICD",
    "SNLBCPT",
    "ALNDCPT",
    "PARTIALCPT",
    "SUBSIMPLECPT",
    "RADICALCPT",
    "MODIFIEDRADICALCPT",
    "IMMEDIATECPT",
    "DELAYEDCPT",
    "TEINSERTIONCPT",
    "TEEXPANDERCPT",
    "FREECPT",
    "LATCPT",
    "SINTRAMCPT",
    "SINTRAMSUPERCPT",
    "BITRAMCPT",
    "MASTOCPT",
    "BREASTREDCPT",
    "FATGRAFTCPT",
    "ADJTISTRANSCPT",
    "AUGPROSIMPCPT",
    "OTHERRECONTECHCPT",
    "REVRECBREASTCPT",
    "NPWTCPT",
    "URGENCY",
    "ANESTHES",
    "SURGSPEC",
    "INOUT",
]
yr_cohorts = {
    "08_23": list(range(2008, 2023)),
    "14_23": list(range(2014, 2023)),
    "19_23": list(range(2019, 2023)),
    "21_23": list(range(2021, 2023)),
}

## Pre-processing pipeline

In [ ]:
imp_dir = BASE_PATH / "data/raw/cleaned"

df_dict = {
    cohort: pd.read_parquet(imp_dir / f"NSQIP_mast_combined_{cohort}.parquet")
    for cohort in cohort_list
}

Transform dev (used for experimentation)

In [ ]:
data_path = BASE_PATH / "data/prelim/processed"
pipeline_path = BASE_PATH / "data/prelim/pipelines"

for cohort, cohort_df in df_dict.items():
    print(f"{cohort}...")
    for yr_rng, yr_list in yr_cohorts.items():
        print(yr_rng)
        for outcome_name in outcome_list:
            print(f"\t {outcome_name}...")
            transform_export_data_dev(
                df=cohort_df,
                x_cols=x_cols,
                target_col_name=outcome_name,
                data_path=data_path / cohort / yr_rng,
                pipeline_path=pipeline_path / cohort / yr_rng,
                tst_yr=2023,
                include_yrs=yr_list,
            )

## Feature engineering
    

Create commands

In [ ]:
def make_cmd(
    outcome,
    cohort,
    n_cv_folds,
    num_optuna_trials,
    num_perm_repeats,
    n_reduction_repeats,
    perc_per_iter,
    seed,
    n_cv_jobs,
    imp_dir,
    export_dir,
):
    cmd_str = f"export PYTHONPATH={BASE_PATH}; \
                export OMP_NUM_THREADS={n_cv_jobs*2}; \
                uv run python -m src.feat_eng \
                --outcome {outcome} \
                --cohort {cohort} \
                --imp_dir {str(imp_dir)} \
                --export_dir {str(export_dir)} \
                --n_cv_folds {n_cv_folds} \
                --num_optuna_trials {num_optuna_trials} \
                --num_perm_repeats {num_perm_repeats} \
                --n_reduction_repeats {n_reduction_repeats} \
                --perc_per_iter {perc_per_iter} \
                --seed {seed} \
                --n_cv_jobs {n_cv_jobs}"
    return " ".join(cmd_str.split())

In [ ]:
SWARM_DIR = BASE_PATH / "swarm" / "feat_red"
N_THREADS = 3
all_cmds = []
for cohort in cohort_list:
    for yr_rng in yr_cohorts.keys():
        swarm_path = SWARM_DIR / "commands" / yr_rng / f"{cohort}.swarm"
        if swarm_path.exists():
            swarm_path.unlink()
        swarm_path.parent.mkdir(exist_ok=True, parents=True)
        cohort_cmd_list = []
        for outcome in outcome_list:
            cmd = make_cmd(
                outcome=outcome,
                cohort=cohort,
                n_cv_folds=3,
                num_optuna_trials=100,  # 100
                num_perm_repeats=50,  # 50
                n_reduction_repeats=6,  # 6
                perc_per_iter=0.25,
                seed=SEED,
                n_cv_jobs=N_THREADS,
                imp_dir=BASE_PATH / "data/prelim/processed" / cohort / yr_rng,
                export_dir=BASE_PATH / "feat_eng" / cohort / outcome / yr_rng,
            )
            cohort_cmd_list.append(cmd)
        swarm_path.write_text("\n".join(cohort_cmd_list))
        all_cmds += cohort_cmd_list
print(f"Num commands: {len(all_cmds)}")

In [ ]:
from src.swarm import run_feat_red_swarm

run_feat_red_swarm(
    cohort_list=cohort_list,
    yr_rng_list=list(yr_cohorts.keys()),
    log_base_dir=SWARM_DIR / "logs",
    cmd_dir=SWARM_DIR / "commands",
    n_threads=N_THREADS,
    swarm_time="4:00:00",
    gb=5,
    partition="quick",
)

# Analyze results

## Select feature groups

In [ ]:
from functools import reduce
from src.feat_eng_confirm import select_iter, get_rank_df

In [ ]:
relevant_outcomes = [
    "ANY",
    "SERIOUS",
    "MORT",
    "SSI",
    "UNPLNREOP",
]
yr_rng = "14_23"
cohort = "cancer"
rank_df_list = []
# can use arbitrary outcome here
perm_dir = BASE_PATH / "feat_eng" / cohort / "ANY" / yr_rng / "perm_df"
perm_df_base = pd.read_csv(perm_dir / "iter_0.tsv", sep="\t", index_col=0)
all_feats = perm_df_base["Feature"].to_list()

for i, outcome in enumerate(relevant_outcomes):
    if outcome == "MORT":
        ## higher threshold bc of instability
        threshold_perc = 0.25
    else:
        threshold_perc = 0.05
    ## Get best iter
    best_iter = select_iter(
        outcome=outcome,
        yr_rng=yr_rng,
        cohort=cohort,
        num_bins=3,
        iter_list=[f"iter_{i}" for i in range(6)][::-1],
        threshold_perc=threshold_perc,
    )
    print(f"Outcome: {outcome} \t Iter: {best_iter}")
    ## Get corresponding features
    perm_df_best = pd.read_csv(perm_dir / f"{best_iter}.tsv", sep="\t", index_col=0)
    rank_df = get_rank_df(perm_df_best, all_feats, outcome)
    rank_df_list.append(rank_df)

full_rank_df = reduce(
    lambda left, right: pd.merge(left, right, on="Feature", how="inner"), rank_df_list
)
rank_cols = [f"Rank_{o}" for o in relevant_outcomes]
full_rank_df["Rank_mean"] = full_rank_df[rank_cols].mean(axis=1)
full_rank_df = full_rank_df.sort_values(by="Rank_mean", ascending=False).reset_index(
    drop=True
)

In [ ]:
display(full_rank_df[["Feature", "Rank_mean"]])

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(full_rank_df["Rank_mean"].values)
plt.xlabel("Feature index (sorted by mean rank)")
plt.ylabel("Mean rank")
plt.title("Consensus feature importance — elbow plot")
plt.xticks(range(len(full_rank_df)), rotation=90, fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
feat_dict = {
    "small": full_rank_df.iloc[:13]["Feature"],
    "moderate": full_rank_df.iloc[:17]["Feature"],
    "big": full_rank_df.iloc[:30]["Feature"],
}

## Confirm performance

- BIG feature set was selected

In [ ]:
def make_confirm_cmd(
    outcome,
    imp_dir,
    export_dir,
    selected_groups,
    n_cv_folds,
    num_optuna_trials,
    seed,
    n_cv_jobs,
):
    cmd_str = f"export PYTHONPATH={BASE_PATH}; \
                export OMP_NUM_THREADS={n_cv_jobs*2}; \
                uv run python -m src.feat_eng_confirm \
                --outcome {outcome} \
                --imp_dir {str(imp_dir)} \
                --export_dir {str(export_dir)} \
                --selected_groups {' '.join(selected_groups)} \
                --n_cv_folds {n_cv_folds} \
                --num_optuna_trials {num_optuna_trials} \
                --seed {seed} \
                --n_cv_jobs {n_cv_jobs}"
    return " ".join(cmd_str.split())

In [ ]:
all_cmds = []
cohort = "cancer"
yr_rng = "14_23"
SWARM_DIR = BASE_PATH / "swarm" / f"feat_red_confirm_{yr_rng}"
cmd_base_dir = SWARM_DIR / "commands"
N_THREADS = 3
for group_size, selected_group in feat_dict.items():
    sub_cmd_list = []
    swarm_path = cmd_base_dir / f"{group_size}.swarm"
    if swarm_path.exists():
        swarm_path.unlink()
    swarm_path.parent.mkdir(exist_ok=True, parents=True)
    for outcome in outcome_list:
        cmd = make_confirm_cmd(
            outcome=outcome,
            imp_dir=BASE_PATH / "data/prelim/processed" / cohort / yr_rng,
            export_dir=BASE_PATH
            / "feat_eng_confirm"
            / f"{cohort}_{yr_rng}"
            / outcome
            / group_size,
            selected_groups=selected_group,
            n_cv_folds=3,
            num_optuna_trials=150,
            seed=SEED,
            n_cv_jobs=N_THREADS,
        )
        sub_cmd_list.append(cmd)
    swarm_path.write_text("\n".join(sub_cmd_list))
    all_cmds += sub_cmd_list
print(f"Num commands: {len(all_cmds)}")

In [ ]:
from src.swarm import run_feat_red_confirm_swarm

run_feat_red_confirm_swarm(
    group_size_list=list(feat_dict.keys()),
    log_base_dir=SWARM_DIR / "logs",
    cmd_dir=cmd_base_dir,
    n_threads=N_THREADS,
    swarm_time="0:30:00",
    gb=3,
    partition="quick",
)

In [ ]:
base_confirm_dir = BASE_PATH / "feat_eng_confirm/cancer_14_23"
analysis_rows = []
for outcome_dir in base_confirm_dir.iterdir():
    outcome_name = outcome_dir.name
    new_row = {
        "outcome": outcome_name,
    }
    for feat_set_dir in outcome_dir.iterdir():
        feat_set_name = feat_set_dir.name
        imp_dir = feat_set_dir / outcome_name
        metric_df = pd.read_csv(imp_dir / "metrics.tsv", sep="\t", index_col=0)
        bin_df = pd.read_csv(imp_dir / "bins/bin_report_3.tsv", sep="\t", index_col=0)
        new_row.update(
            {
                "event_rate": metric_df["event_rate"].iloc[0],
                f"AP_lift_{feat_set_name}": metric_df["test_ap (lift)"]
                .iloc[0]
                .split("(")[1]
                .strip("()"),
                f"AUROC_{feat_set_name}": float(metric_df["test_auroc"].iloc[0]),
                f"bin_lift_{feat_set_name}": float(bin_df.loc["High", "lift"]),
            }
        )
    analysis_rows.append(new_row)
analysis_df = pd.DataFrame(analysis_rows)
analysis_df